# Strands Agents with Bedrock AgentCore Code Interpreter

This workshop demonstrates how to integrate Strands Agents with Amazon Bedrock AgentCore Code Interpreter to create AI agents capable of executing code dynamically.

## Overview

In this lab, you will:
- Learn about Bedrock AgentCore Code Interpreter capabilities
- Test the default Code Interpreter with Strands Agents
- Create a custom Code Interpreter with public network access
- Compare different execution environments and their limitations

## Prerequisites

Before starting this lab, ensure you have:
- AWS credentials configured (IAM role or environment variables)
- Required Python packages installed
- Nova Pro model ID based on AWS region

If you're not running in an environment with an IAM role assumed, set your AWS credentials as environment variables:

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"]=<YOUR ACCESS KEY>
#os.environ["AWS_SECRET_ACCESS_KEY"]=<YOUR SECRET KEY>
#os.environ["AWS_SESSION_TOKEN"]=<OPTIONAL - YOUR SESSION TOKEN IF TEMP CREDENTIAL>
#os.environ["AWS_REGION"]=<AWS REGION WITH BEDROCK AGENTCORE AVAILABLE>

Install required packages for Strands Agents and Bedrock AgentCore Python SDK:

In [1]:
%pip install -q strands-agents strands-agents-tools bedrock-agentcore rich

/workshop/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


Setup Nova Pro model ID based on AWS region:

In [2]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Nova Pro Model ID: us.amazon.nova-pro-v1:0


## What is Bedrock AgentCore Code Interpreter?

Amazon Bedrock AgentCore Code Interpreter is a powerful tool that allows AI agents to execute code dynamically in a secure sandbox environment. Key benefits include:

- **Secure Execution**: Runs Python code in an isolated sandbox environment
- **Dynamic Problem Solving**: Enables agents to perform calculations, analyze data, and generate visualizations
- **Flexible Configuration**: Supports both default sandboxed and custom network-enabled environments
- **Integration Ready**: Seamlessly integrates with Strands Agents and other AI frameworks

The Code Interpreter provides agents with the ability to solve complex problems that require computational analysis, data processing, or mathematical calculations.

### Testing Strands Agent with Default Code Interpreter

Let's start by testing the Strands Agent using the default AgentCore Code Interpreter. We'll demonstrate how it can generate and execute code to solve mathematical problems in a sandboxed environment.

In [3]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

agentcore_code_interpreter = AgentCoreCodeInterpreter()

# Create a code-gen assistant agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="""You are a helpful AI assistant that validates all answers through code execution.""",
    tools=[agentcore_code_interpreter.code_interpreter],
)

agent("What is the area of a circle with radius 8.26cm?")

<thinking> To find the area of a circle, I can use the formula \( \text{Area} = \pi r^2 \), where \( r \) is the radius of the circle. In this case, the radius is 8.26 cm. I will use the code interpreter tool to calculate the area. </thinking>


Tool #1: code_interpreter
<thinking> It seems that the session for calculating the area of the circle was not found. I need to create a new session and then execute the code to calculate the area. </thinking> 
Tool #2: code_interpreter

Tool #3: code_interpreter
<thinking> It seems that there was an issue with the session again. I will try to create the session again and execute the code to calculate the area. </thinking> 
Tool #4: code_interpreter

Tool #5: code_interpreter
<thinking> The session now exists, and I have successfully calculated the area of the circle. The result is approximately 214.34 square centimeters. I will now provide this information to the user. </thinking>

The area of a circle with a radius of 8.26 cm is approximately 

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '<thinking> The session now exists, and I have successfully calculated the area of the circle. The result is approximately 214.34 square centimeters. I will now provide this information to the user. </thinking>\n\nThe area of a circle with a radius of 8.26 cm is approximately 214.34 square centimeters.'}]}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'code_interpreter': ToolMetrics(tool={'toolUseId': 'tooluse_4Xej9Yi5MtLicxWQVYkI9m', 'name': 'code_interpreter', 'input': {'code_interpreter_input': {'action': {'session_name': 'circle_area_session', 'language': 'python', 'code': 'import math\nradius = 8.26\narea = math.pi * (radius ** 2)\nprint(area)', 'type': 'executeCode'}}}}, call_count=5, success_count=2, error_count=3, total_time=1.0578784942626953)}, cycle_durations=[1.646350383758545], traces=[<strands.telemetry.metrics.Trace object at 0xffff73c0ef50>, <strands.telemetry.metrics.Trace o

Let's examine the detailed execution flow of the agent loop to understand how the agent processes requests and generates responses:

In [4]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

Agent Loop Detail

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Number of Loops: 4

                                                  Agent Messages                                                   
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Text                      ┃ Tool Name        ┃ Tool Input                ┃ Tool Result              ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user      │ What is the area of a     │                  │                           │                          │
│           │ circle with radius        │                  │                           │                          │
│           │ 8.26cm?                   │                  │                           │                          │
├───────────┼───────────────────────────┼──────────────────┼───────────────────────────┼──────────────────────────┤
│ assistant │ <thinking> To find the    │ code_interpreter │ {                         │                          │
│           │ area of a circle, I can   │                  │   "code_interpreter_inpu… │                          │
│           │ use the formula \(        │                  │ {                         │                          │
│           │ \text{Area} = \pi r^2 \), │                  │     "action": {           │                          │
│           │ where \( r \) is the      │                  │       "type":             │                          │
│           │ radius of the circle. In  │                  │ "executeCode",            │                          │
│           │ this case, the radius is  │                  │       "session_name":     │                          │
│           │ 8.26 cm. I will use the   │                  │ "circle_area_session",    │                          │
│           │ code interpreter tool to  │                  │       "code": "import     │                          │
│           │ calculate the area.       │                  │ math\nradius = 8.26\narea │                          │
│           │ </thinking>               │                  │ = math.pi * (radius **    │                          │
│           │                           │                  │ 2)\nprint(area)",         │                          │
│           │                           │                  │       "language":         │                          │
│           │                           │                  │ "python"                  │                          │
│           │                           │                  │     }                     │                          │
│           │                           │                  │   }                       │                          │
│           │                           │                  │ }                         │                          │
├───────────┼───────────────────────────┼──────────────────┼───────────────────────────┼──────────────────────────┤
│ user      │                           │                  │                           │ {                        │
│           │                           │                  │                           │   "text": "Session       │
│           │                           │                  │                           │ 'circle_area_session'    │
│           │                           │                  │                           │ not found"               │
│           │                           │                  │                           │ }                        │
├───────────┼───────────────────────────┼──────────────────┼───────────────────────────┼──────────────────────────┤
│ assistant │ <thinking> It seems that  │ code_interpreter │ {                         │                          │
│           │ the session for           │                  │   "code_interpreter_inpu… │                          │
│           │ calculating the area of   │               

### Understanding Sandbox Environment Limitations

The default Code Interpreter operates in a **sandbox environment with network isolation**. This is an important security feature that prevents unauthorized network access while ensuring code execution safety.

## Creating a Custom Code Interpreter with Network Access

To enable web-based operations, we'll create a custom Code Interpreter with public network access. This demonstrates the flexibility of the AgentCore platform for different use cases.

### Step 1: Initialize AgentCore Clients

First, let's set up the necessary clients for both control plane and data plane operations:

In [5]:
from bedrock_agentcore._utils import endpoints
import boto3
import json

region = boto3.session.Session().region_name

data_plane_endpoint = endpoints.get_data_plane_endpoint(region)
control_plane_endpoint = endpoints.get_control_plane_endpoint(region)

cp_client = boto3.client("bedrock-agentcore-control", 
                        region_name=region,
                        endpoint_url=control_plane_endpoint)

dp_client = boto3.client("bedrock-agentcore", 
                        region_name=region,
                        endpoint_url=data_plane_endpoint)

### Step 2: Create Custom Code Interpreter

Create a custom Code Interpreter with public network access:

In [6]:
from botocore.exceptions import ClientError

interpreter_name = "SampleCodeInterpreter"

# Create code interpreter
try:
    interpreter_response = cp_client.create_code_interpreter(
        name=interpreter_name,
        description="Environment for Code Interpreter sample test",
        #executionRoleArn=iam_role_arn, #Required only if the code interpreter need to access AWS resources
        networkConfiguration={
            'networkMode': 'PUBLIC'
        }
    )
    interpreter_id = interpreter_response["codeInterpreterId"]
    print(f"Created interpreter: {interpreter_id}")
except ClientError as e:
    print(f"ERROR: {e}")
    if "already exists" in str(e):
        # If code interpreter already exists, retrieve its ID
        for items in cp_client.list_code_interpreters()['codeInterpreterSummaries']:
            if items['name'] == interpreter_name:
                interpreter_id = items['codeInterpreterId']
                print(f"Code Interpreter ID: {interpreter_id}")
                break
except Exception as e:
    # Show any errors during code interpreter creation
    print(f"ERROR: {e}")

Created interpreter: SampleCodeInterpreter-LxbtYjo6Qg


### Step 3: Create Code Interpreter Session

Create a code interpreter session in the custom Code Interpreter:

In [7]:
from botocore.exceptions import ClientError

session_name = "SampleCodeInterpreterSession"

# Create code interpreter session
session_response = dp_client.start_code_interpreter_session(
    codeInterpreterIdentifier=interpreter_id,
    name=session_name,
    sessionTimeoutSeconds=900
)
session_id = session_response["sessionId"]
print(f"Created session: {session_id}")

Created session: 01KKRC0KRA3J0DCRV260Q017J5


### Step 4: Test Basic Functionality

Let's verify if the code interpreter session can access the internet by installing the Yahoo Finance Python package with pip and using the package to retrieve Amazon stock price today:

In [8]:
response = dp_client.invoke_code_interpreter(
    codeInterpreterIdentifier=interpreter_id,
    sessionId=session_id,
    name="executeCommand",
    arguments={
        'command': "pip install yfinance"
    }
)
response = dp_client.invoke_code_interpreter(
    codeInterpreterIdentifier=interpreter_id,
    sessionId=session_id,
    name="executeCode",
    arguments={"code": """
        import yfinance as yf
               
        amzn = yf.Ticker('AMZN')
        data = amzn.history(period='1d')
        today_close = data['Close'][-1]
        print(today_close)
        """,
        "language": "python",
        "clearContext": False
    }
)
for event in response["stream"]:
    print(json.dumps(event["result"], indent=2))

{
  "content": [
    {
      "type": "text",
      "text": "207.6699981689453"
    },
    {
      "type": "text",
      "text": "/tmp/ipykernel_299/3027036176.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`\n  today_close = data['Close'][-1]"
    }
  ],
  "structuredContent": {
    "stdout": "207.6699981689453",
    "stderr": "/tmp/ipykernel_299/3027036176.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`\n  today_close = data['Close'][-1]",
    "exitCode": 0,
    "executionTime": 1.6829545497894287
  },
  "isError": false
}


## Using Custom Code Interpreter with Strands Agent

Now we'll integrate the custom Code Interpreter with internet access into Strands Agent. We'll create custom `execute_python` and `execute_command` tools that share the same code interpreter session for enhanced functionality.

In [9]:
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.tools.code_interpreter_client import CodeInterpreter
import boto3

#Reuse the Core Interpreter and Session created above
ci_client = CodeInterpreter(region=boto3.session.Session().region_name)
ci_client.start(identifier=interpreter_id) #initializes a new code interpreter session

@tool
def execute_python(code: str, description: str = "") -> str:
    """Execute Python code in the sandbox."""
    
    print(f"\n Generated Code: {code}")
    response = ci_client.invoke("executeCode", {
        "code": code,
        "language": "python",
        "clearContext": False
    })
        
    for event in response["stream"]:
        return json.dumps(event["result"])

@tool
def execute_command(command: str, description: str = "") -> str:
    """Execute command in the sandbox."""
    
    print(f"\n Generated Command: {command}")
    response = ci_client.invoke("executeCommand", {
        "command": command
    })
    
    for event in response["stream"]:
        return json.dumps(event["result"])
    

# Create a code gen agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="""You are a helpful AI assistant that validates all answers through code execution.
                  If you has no available tools to perform the task, you must generate and execute code to continue. 
                  If required, execute pip install to download required packages.
                  """,
    tools=[execute_python, execute_command],
)

agent("What is the stock price of Amazon today?")

ci_client.stop() #stop the current code interpreter session

<thinking> To obtain the stock price of Amazon, I need to use an API that provides real-time stock market data. One such API is the Alpha Vantage API, which offers free access to real-time and historical stock market data. First, I need to install the `alpha_vantage` package if it is not already installed. Then, I can use the API to fetch the stock price. </thinking>


Tool #1: execute_command

 Generated Command: pip install alpha_vantage
<thinking> The `alpha_vantage` package has been successfully installed. Now, I need to fetch the stock price of Amazon using the Alpha Vantage API. I will write a Python script to do this. First, I need to get an API key from Alpha Vantage, but for demonstration purposes, I will use a placeholder key. In a real-world scenario, you would need to replace this with your own API key. </thinking>

Tool #2: execute_python

 Generated Code: from alpha_vantage.timeseries import TimeSeries

# Replace 'YOUR_API_KEY' with your actual Alpha Vantage API key
api_k

True

Let's examine the detailed execution flow of the agent loop to understand how the agent processes requests and generates responses:

In [10]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

Agent Loop Detail

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Number of Loops: 4

                                                  Agent Messages                                                   
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role      ┃ Text                      ┃ Tool Name       ┃ Tool Input                ┃ Tool Result               ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user      │ What is the stock price   │                 │                           │                           │
│           │ of Amazon today?          │                 │                           │                           │
├───────────┼───────────────────────────┼─────────────────┼───────────────────────────┼───────────────────────────┤
│ assistant │ <thinking> To obtain the  │ execute_command │ {                         │                           │
│           │ stock price of Amazon, I  │                 │   "description": "Install │                           │
│           │ need to use an API that   │                 │ the alpha_vantage         │                           │
│           │ provides real-time stock  │                 │ package.",                │                           │
│           │ market data. One such API │                 │   "command": "pip install │                           │
│           │ is the Alpha Vantage API, │                 │ alpha_vantage"            │                           │
│           │ which offers free access  │                 │ }                         │                           │
│           │ to real-time and          │                 │                           │                           │
│           │ historical stock market   │                 │                           │                           │
│           │ data. First, I need to    │                 │                           │                           │
│           │ install the               │                 │                           │                           │
│           │ `alpha_vantage` package   │                 │                           │                           │
│           │ if it is not already      │                 │                           │                           │
│           │ installed. Then, I can    │                 │                           │                           │
│           │ use the API to fetch the  │                 │                           │                           │
│           │ stock price. </thinking>  │                 │                           │                           │
│           │                           │                 │                           │                           │
│           │                           │                 │                           │                           │
├───────────┼───────────────────────────┼─────────────────┼───────────────────────────┼───────────────────────────┤
│ user      │                           │                 │                           │ {                         │
│           │                           │                 │                           │   "text": "{\"content\":  │
│           │                           │                 │                           │ [{\"type\": \"text\",     │
│           │                           │                 │                           │ \"text\": \"Collecting    │
│           │                           │                 │                           │ alpha_vantage\\r\\n       │
│           │                           │                 │                           │ Downloading               │
│           │                           │                 │                           │ alpha_vantage-3.0.0-py3-… │
│           │                           │                 │                           │ (12 kB)\\r\\nCollecting   │
│           │                           │               

## Resource Cleanup (Optional)

Clean up the AgentCore Runtime resources we created to avoid unnecessary charges:

In [11]:
import boto3

cp_client = boto3.client("bedrock-agentcore-control", region_name=region, endpoint_url=control_plane_endpoint)
dp_client = boto3.client("bedrock-agentcore", region_name=region, endpoint_url=data_plane_endpoint)

try:
    print("Cleaning up session and interpreter...")
    dp_client.stop_code_interpreter_session(
        codeInterpreterIdentifier=interpreter_id,
        sessionId=session_id
    )
    print("✓ Session stopped successfully")

    cp_client.delete_code_interpreter(codeInterpreterId=interpreter_id)
    print("✓ Interpreter deleted successfully")
except Exception as e:
    print(f"❌ Error during cleanup: {e}")
    print("You may need to manually clean up some resources.")

Cleaning up session and interpreter...
✓ Session stopped successfully
✓ Interpreter deleted successfully


## Conclusion

In this lab, you successfully:

- ✅ Tested default Bedrock AgentCore Code Interpreter functionality
- ✅ Created a custom Code Interpreter with network access capabilities
- ✅ Integrated Code Interpreter with Strands Agents for dynamic Python execution
- ✅ Executed complex tasks including data analysis, web requests, and file operations

## Key Benefits of AgentCore Code Interpreter

- **Dynamic Code Execution**: Run Python code on-demand within AI agent workflows
- **Secure Sandbox Environment**: Isolated execution with configurable network access
- **Session State Management**: Maintain variables and context across multiple executions
- **Seamless Agent Integration**: Works as a native tool within Strands Agents
- **Flexible Configuration**: Customize interpreters for specific use cases and requirements